## Difix3D+ Post-processing and ablation study (OOD-NVS)

## Step 1 - Env set-up

In [ ]:
import os, subprocess


CONDA_PREFIX = "/usr/local/miniconda"
CONDA_BIN = f"{CONDA_PREFIX}/bin/conda"
DIFIX_ENV_PREFIX = f"{CONDA_PREFIX}/envs/difix"
DIFIX_PYTHON_BIN = f"{DIFIX_ENV_PREFIX}/bin/python"
DIFIX_PIP_BIN = f"{DIFIX_ENV_PREFIX}/bin/pip"

env = os.environ.copy()
env["CUDA_HOME"] = "/usr/local/cuda-11.8"
env["PATH"] = f"/usr/local/cuda-11.8/bin:{DIFIX_ENV_PREFIX}/bin:{CONDA_PREFIX}/bin:" + env.get("PATH", "")
env["CC"] = "/usr/bin/gcc-11"
env["CXX"] = "/usr/bin/g++-11"
env["CUDAHOSTCXX"] = "/usr/bin/g++-11"
env["TORCH_CUDA_ARCH_LIST"] = "7.5;8.0;8.6"


if not os.path.isfile(DIFIX_PYTHON_BIN):
    print("-> Tạo môi trường conda 'difix' (Python 3.10) từ conda-forge...")
    cmd_create = f"{CONDA_BIN} create -y -n difix -c conda-forge --override-channels python=3.10 pip wheel 'setuptools<70.0.0'"
    r = subprocess.run(
        cmd_create,
        shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    if r.returncode != 0:
        raise RuntimeError("Tạo conda env 'difix' thất bại:\n" + r.stdout[-3000:])
else:
    print("-> Môi trường conda 'difix' đã có sẵn, bỏ qua tạo env.")

check_torch = subprocess.run(
    [DIFIX_PYTHON_BIN, "-c", "import torch; assert torch.__version__.startswith('2.1.2') and torch.cuda.is_available(); print(torch.__version__, 'CUDA:', torch.cuda.is_available())"],
    capture_output=True, text=True, env=env
)

if check_torch.returncode != 0:
    print("-> Cài PyTorch 2.1.2 (cu118) trong env 'difix'...")
    r = subprocess.run(
        f"{DIFIX_PIP_BIN} install torch==2.1.2 torchvision==0.16.2 torchaudio==2.1.2 "
        f"--index-url https://download.pytorch.org/whl/cu118",
        shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env
    )
    if r.returncode != 0:
        raise RuntimeError(" Cài PyTorch trong env 'difix' thất bại:\n" + r.stdout[-3000:])

    verify = subprocess.run(
        [DIFIX_PYTHON_BIN, "-c", "import torch; print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())"],
        capture_output=True, text=True, env=env
    )
    print(f"-> Đã cài xong: {verify.stdout.strip()}")
else:
    print(f"-> PyTorch đã chuẩn trong env 'difix' ({check_torch.stdout.strip()}), bỏ qua.")

print("\n Xong BƯỚС 8-difix(a)!")

=== BƯỚС 8-difix(a): TẠO ENV CONDA 'difix' + CÀI PYTORCH ===
-> Tạo môi trường conda 'difix' (Python 3.10) từ conda-forge...
-> Cài PyTorch 2.1.2 (cu118) trong env 'difix'...
-> Đã cài xong: PyTorch: 2.1.2+cu118 | CUDA available: True

✅ Xong BƯỚС 8-difix(a)!


In [ ]:
import os, subprocess

print("=== BƯỚС 8-difix(b): CLONE Difix3D + CÀI DEPENDENCIES ===")

DIFIX_REPO = "/content/Difix3D"
DIFIX_ENV_PREFIX = "/usr/local/miniconda/envs/difix"
DIFIX_PYTHON_BIN = f"{DIFIX_ENV_PREFIX}/bin/python"
DIFIX_PIP_BIN = f"{DIFIX_ENV_PREFIX}/bin/pip"

if not os.path.isdir(DIFIX_REPO):
    print("-> Đang clone Difix3D (recursive)...")
    r = subprocess.run(
        ["git", "clone", "--recursive", "https://github.com/nv-tlabs/Difix3D.git", DIFIX_REPO],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        raise RuntimeError(f"Clone Difix3D thất bại:\n{r.stderr}")
    print(f"-> Đã clone thành công {DIFIX_REPO}")
else:
    print(f"-> {DIFIX_REPO} đã tồn tại. Đang cập nhật submodules nếu thiếu...")
    subprocess.run(["git", "-C", DIFIX_REPO, "submodule", "update", "--init", "--recursive"],
                   capture_output=True, text=True)

req_path = os.path.join(DIFIX_REPO, "requirements.txt")
reqs = []
if os.path.isfile(req_path):
    with open(req_path) as f:
        for line in f:
            clean = line.strip()

            if clean and not clean.startswith("#") and not clean.lower().startswith(("torch", "torchvision", "torchaudio")):
                reqs.append(clean)

extra_pkgs = ["tensorboard", "accelerate", "transformers", "lpips", "jaxtyping", "imageio", "imageio-ffmpeg"]
for p in extra_pkgs:
    if p not in reqs:
        reqs.append(p)

check = subprocess.run(
    [DIFIX_PYTHON_BIN, "-c", "import diffusers, peft, lpips, tensorboard, accelerate"],
    capture_output=True, text=True
)

if check.returncode != 0:
    print(f"-> Đang cài đặt {len(reqs)} thư viện (đã khóa cứng PyTorch 2.1.2 cu118)...")

    cmd_install = [DIFIX_PIP_BIN, "install"] + reqs
    r = subprocess.run(cmd_install, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"pip install requirements thất bại:\nSTDOUT:\n{r.stdout[-2000:]}\nSTDERR:\n{r.stderr[-2000:]}")

    verify_torch = subprocess.run(
        [DIFIX_PYTHON_BIN, "-c", "import torch; assert torch.__version__.startswith('2.1.2') and torch.cuda.is_available(); print(torch.__version__, 'CUDA:', torch.cuda.is_available())"],
        capture_output=True, text=True
    )
    if verify_torch.returncode != 0:
        print("⚠️ Cảnh báo: PyTorch bị ghi đè phiên bản! Đang khôi phục lại torch 2.1.2+cu118...")
        subprocess.run(
            f"{DIFIX_PIP_BIN} install --force-reinstall --no-deps torch==2.1.2 torchvision==0.16.2 "
            f"--index-url https://download.pytorch.org/whl/cu118",
            shell=True, check=True
        )
    print(" Đã cài đặt xong toàn bộ dependencies an toàn.")
else:
    print("-> Tất cả thư viện lõi (diffusers, peft, lpips, accelerate, tensorboard) đã sẵn sàng trong env 'difix'.")

r_status = subprocess.run(
    [DIFIX_PYTHON_BIN, "-c", "import torch, diffusers; print(f'Ready: PyTorch {torch.__version__} (CUDA: {torch.cuda.is_available()}), Diffusers {diffusers.__version__}')"],
    capture_output=True, text=True
)
print(f"-> {r_status.stdout.strip()}")
print("\n Xong BƯỚС 8-difix(b)!")

=== BƯỚС 8-difix(b): CLONE Difix3D + CÀI DEPENDENCIES ===
-> Đang clone Difix3D (recursive)...
-> Đã clone thành công /content/Difix3D
-> Đang cài đặt 15 thư viện (đã khóa cứng PyTorch 2.1.2 cu118)...
⚠️ Cảnh báo: PyTorch bị ghi đè phiên bản! Đang khôi phục lại torch 2.1.2+cu118...
✅ Đã cài đặt xong toàn bộ dependencies an toàn.
-> Ready: PyTorch 2.1.2+cu118 (CUDA: True), Diffusers 0.25.1

✅ Xong BƯỚС 8-difix(b)!


## Step 2 - Paired dataset (Splatformer renders vs ground truth)

In [ ]:
import os, glob, json, random
from PIL import Image

print("=== BƯỚС 8-difix(c): TẠO data.json (Render <-> GT) CHO FINE-TUNE DIFIX ===")

DIFIX_DATA_JSON = "/content/Difix3D/data/mvsgaussian_data.json"
os.makedirs(os.path.dirname(DIFIX_DATA_JSON), exist_ok=True)

DIFIX_PROMPT = "remove degradation"
DIFIX_TEST_FRACTION = 0.1  # 10% test, 90% train

pairs = []

splatformer_dir = "/content/SplatFormer/outputs/ood_eval_mvsinit_alpha/test/default"
if os.path.exists(splatformer_dir):
    pred_files = sorted(glob.glob(os.path.join(splatformer_dir, "*_pred.png")))
    for pred_p in pred_files:
        gt_p = pred_p.replace("_pred.png", "_gt.png")
        if os.path.exists(gt_p):
            pairs.append((pred_p, gt_p))

preview_dir = "/content/alpha_preview_all"
if os.path.exists(preview_dir):
    splat_preds = sorted(glob.glob(os.path.join(preview_dir, "*_pred.png")))
    for pred_p in splat_preds:
        gt_p = pred_p.replace("_pred.png", "_gt.png")
        if os.path.exists(gt_p) and (pred_p, gt_p) not in pairs:
            pairs.append((pred_p, gt_p))

composite_images = glob.glob("/content/SplatFormer/outputs/ood_eval_mvsinit_alpha/**/*.png", recursive=True)
composite_images = [p for p in composite_images if "default" not in p and "test" in p]

if len(pairs) == 0 and composite_images:
    print("-> Phát hiện ảnh dạng composite ghép ngang, đang tự động tách cột...")
    sliced_dir = "/content/difix_sliced_pairs"
    os.makedirs(sliced_dir, exist_ok=True)
    for idx, cp in enumerate(composite_images):
        img = Image.open(cp).convert("RGB")
        w, h = img.size
        if w >= 3 * h:
            step = w // 3
            pred_img = img.crop((step, 0, 2 * step, h))
            gt_img = img.crop((2 * step, 0, 3 * step, h))

            p_path = os.path.join(sliced_dir, f"pair_{idx:04d}_pred.png")
            g_path = os.path.join(sliced_dir, f"pair_{idx:04d}_gt.png")
            pred_img.save(p_path)
            gt_img.save(g_path)
            pairs.append((p_path, g_path))

print(f"-> Tổng số cặp (Pred <-> GT) thu thập được: {len(pairs)}")

if not pairs:
    raise RuntimeError(
        "Không tìm thấy cặp ảnh render/GT nào! "
        "Hãy chắc chắn rằng BƯỚС 5-ood(b) hoặc render preview đã hoàn tất."
    )

random.Random(42).shuffle(pairs)
n_test = max(1, int(len(pairs) * DIFIX_TEST_FRACTION))
test_pairs = pairs[:n_test]
train_pairs = pairs[n_test:]

if len(train_pairs) == 0:
    train_pairs = test_pairs.copy()

def to_split_dict(pair_list):
    return {
        f"{i:05d}": {
            "image": os.path.abspath(inp),
            "target_image": os.path.abspath(tgt),
            "prompt": DIFIX_PROMPT
        }
        for i, (inp, tgt) in enumerate(pair_list)
    }

data_json = {
    "train": to_split_dict(train_pairs),
    "test": to_split_dict(test_pairs)
}

with open(DIFIX_DATA_JSON, "w") as f:
    json.dump(data_json, f, indent=2)

print(f"\nXong BƯỚС 8-difix(c):")
print(f"  * Số cặp Train: {len(train_pairs)}")
print(f"  * Số cặp Test : {len(test_pairs)}")
print(f"  * File JSON đã tạo: {DIFIX_DATA_JSON}")

=== BƯỚС 8-difix(c): TẠO data.json (Render <-> GT) CHO FINE-TUNE DIFIX ===
-> Tổng số cặp (Pred <-> GT) thu thập được: 20

✅ Xong BƯỚС 8-difix(c):
  * Số cặp Train: 18
  * Số cặp Test : 2
  * File JSON đã tạo: /content/Difix3D/data/mvsgaussian_data.json


## Step 3 - Source code patches (resolution alignment and pretrained weight loading)

In [ ]:
import os, re

dataset_py = "/content/Difix3D/src/dataset.py"
with open(dataset_py, "r") as f:
    code = f.read()

# Replace with height=512, width=512
code_patched = re.sub(
    r"def __init__\(self,\s*dataset_path,\s*split,\s*height=\d+,\s*width=\d+",
    "def __init__(self, dataset_path, split, height=512, width=512",
    code
)

with open(dataset_py, "w") as f:
    f.write(code_patched)

print("Đã vá dataset.py về kích thước chuẩn (height=512, width=512) để tối ưu VRAM cho GPU T4.")

✅ Đã vá dataset.py về kích thước chuẩn (height=512, width=512) để tối ưu VRAM cho GPU T4.


### Import pretrained weights from NVDIA training

In [ ]:
import os

print("=== BƯỚС 8-difix(d2): Vá train_difix.py -- nạp pretrained thật trước khi train ===")

TRAIN_PY = "/content/Difix3D/src/train_difix.py"
with open(TRAIN_PY) as f:
    src = f.read()

MARKER = "# [PATCH] nạp thật trọng số pretrained_model_name_or_path"
OLD = """    net_difix = Difix(
        lora_rank_vae=args.lora_rank_vae,
        timestep=args.timestep,
        mv_unet=args.mv_unet,
    )
    net_difix.set_train()"""
NEW = """    net_difix = Difix(
        lora_rank_vae=args.lora_rank_vae,
        timestep=args.timestep,
        mv_unet=args.mv_unet,
    )

    if getattr(args, "pretrained_model_name_or_path", None):
        print(f"-> [PATCH] Nạp trọng số pretrain thật từ '{args.pretrained_model_name_or_path}' "
              f"(thay cho sd-turbo + LoRA rỗng)...")
        from pipeline_difix import DifixPipeline
        _pipe = DifixPipeline.from_pretrained(args.pretrained_model_name_or_path, trust_remote_code=True)
        _n_unet_before = sum(p.numel() for p in net_difix.unet.parameters())
        _n_vae_before = sum(p.numel() for p in net_difix.vae.parameters())
        net_difix.unet = _pipe.unet.to("cuda")
        net_difix.vae = _pipe.vae.to("cuda")
        net_difix.vae.decoder.gamma = 1  # Difix.__init__ set thuộc tính này cho vae gốc, set lại cho vae vừa gán
        _n_unet_after = sum(p.numel() for p in net_difix.unet.parameters())
        _n_vae_after = sum(p.numel() for p in net_difix.vae.parameters())
        print(f"   UNet: {_n_unet_before/1e6:.1f}M tham số (sd-turbo gốc) -> "
              f"{_n_unet_after/1e6:.1f}M tham số (nvidia/difix thật)")
        print(f"   VAE : {_n_vae_before/1e6:.1f}M tham số (random LoRA)   -> "
              f"{_n_vae_after/1e6:.1f}M tham số (nvidia/difix thật)")
        if abs(_n_unet_after - _n_unet_before) < 1000:
            print("   CẢNH BÁO: số tham số UNet gần như không đổi -- nghi swap KHÔNG có tác dụng, "
                  "kiểm tra lại thủ công trước khi tin tưởng kết quả train.")
        del _pipe
        torch.cuda.empty_cache()

    net_difix.set_train()"""

if MARKER in src:
    print(f"-> {TRAIN_PY} đã được vá từ trước (marker tìm thấy), bỏ qua.")
elif OLD in src:
    src = src.replace(OLD, NEW, 1)
    with open(TRAIN_PY, "w") as f:
        f.write(src)
    print(f"Đã vá {TRAIN_PY}: thêm bước nạp thật trọng số pretrained_model_name_or_path.")
else:
    idx = src.find("net_difix = Difix(")
    raise RuntimeError(
        f"Không tìm thấy CẢ marker đã-vá LẪN đoạn code gốc cần vá trong {TRAIN_PY} -- có thể bạn đã "
        f"tự sửa file này trước đó theo cách khác. 400 ký tự quanh 'net_difix = Difix(' để đối chiếu "
        f"thủ công: {repr(src[max(0, idx-100):idx+400]) if idx != -1 else '(không tìm thấy chuỗi đó)'}"
    )

print("\nXong BƯỚС 8-difix(d2). Từ giờ --pretrained_model_name_or_path=nvidia/difix ở BƯỚС8-difix(e) "
      "sẽ thật sự có tác dụng -- xem log 'UNet: ... -> ...' lúc train để tự xác nhận.")

=== BƯỚС 8-difix(d2): Vá train_difix.py -- nạp pretrained thật trước khi train ===
✅ Đã vá /content/Difix3D/src/train_difix.py: thêm bước nạp thật trọng số pretrained_model_name_or_path.

✅ Xong BƯỚС 8-difix(d2). Từ giờ --pretrained_model_name_or_path=nvidia/difix ở BƯỚС8-difix(e) sẽ thật sự có tác dụng -- xem log 'UNet: ... -> ...' lúc train để tự xác nhận.


## Step 4 - Fine-tuning Difix on OOD-NVS renders (384x384, 8-bit AdamW)

In [ ]:
import subprocess

DIFIX_PIP_BIN = "/usr/local/miniconda/envs/difix/bin/pip"

print("-> Gỡ xformers và hạ cấp numpy về <2.0.0...")
subprocess.run(f"{DIFIX_PIP_BIN} uninstall -y xformers", shell=True, check=True)
subprocess.run(f"{DIFIX_PIP_BIN} install -q 'numpy<2.0.0'", shell=True, check=True)

DIFIX_PYTHON = "/usr/local/miniconda/envs/difix/bin/python"
check = subprocess.run(
    [DIFIX_PYTHON, "-c", "import torch, numpy; from diffusers import AutoencoderKL; print('Thành công! NumPy:', numpy.__version__, '| PyTorch:', torch.__version__)"],
    capture_output=True, text=True
)
print(check.stdout)
if check.returncode != 0:
    print("Vẫn còn lỗi:\n", check.stderr)

-> Gỡ xformers và hạ cấp numpy về <2.0.0...
✅ Thành công! NumPy: 1.26.4 | PyTorch: 2.1.2+cu118



In [ ]:
import os, subprocess, re

os.system("cd /content/Difix3D && git checkout src/")

ds_path = "/content/Difix3D/src/dataset.py"
with open(ds_path, "r") as f: code = f.read()
code = re.sub(r"height=\d+,\s*width=\d+", "height=384, width=384", code)

alpha_composite_code = """
        def process_image(img):
            if img.mode == 'RGBA':
                bg = Image.new("RGB", img.size, (255, 255, 255))
                bg.paste(img, mask=img.split()[3])
                return bg
            return img.convert("RGB")

        input_img = process_image(input_img)
        output_img = process_image(output_img)

        img_t = F.to_tensor(input_img)
        output_t = F.to_tensor(output_img)
"""
code = code.replace("img_t = F.to_tensor(img_t)", alpha_composite_code)
code = code.replace("output_t = F.to_tensor(output_t)", "")
with open(ds_path, "w") as f: f.write(code)

md_path = "/content/Difix3D/src/model.py"
with open(md_path, "r") as f: md_code = f.read()
md_code = re.sub(r'(UNet2DConditionModel\.from_pretrained\()[^,]+,', r'\1"nvidia/difix",', md_code)
with open(md_path, "w") as f: f.write(md_code)

tr_path = "/content/Difix3D/src/train_difix.py"
with open(tr_path, "r") as f: code = f.read()

code = "import bitsandbytes as bnb\n" + code
code = code.replace("dataset_val.img_names", "dataset_val.img_ids")
code = code.replace("log_with=args.report_to,", "# log_with=args.report_to,")
code = re.sub(r"(\s+)(accelerator\.clip_grad_norm_.*)", r"\1pass  # \2", code)
code = code.replace("weight_dtype = torch.float16", "weight_dtype = torch.float32")
code = code.replace("torch.optim.AdamW", "bnb.optim.AdamW8bit")

freeze_hook = """
    # --- CỨU CÁNH VRAM CHO TESLA T4 ---
    net_difix.vae.requires_grad_(False)
    net_difix.text_encoder.requires_grad_(False)
    if hasattr(net_difix.vae, 'enable_slicing'):
        net_difix.vae.enable_slicing()
    if hasattr(net_difix.vae, 'enable_tiling'):
        net_difix.vae.enable_tiling()
    # ----------------------------------
"""
lines = code.split('\n')
for i, line in enumerate(lines):
    if "accelerator.prepare(" in line:
        lines.insert(i, freeze_hook)
        break
code = '\n'.join(lines)

with open(tr_path, "w") as f: f.write(code)

print("Đã giảm độ phân giải xuống 384x384 để triệt tiêu hoàn toàn nguy cơ OOM!")

DIFIX_ENV_PREFIX = "/usr/local/miniconda/envs/difix"
ACCELERATE_BIN = f"{DIFIX_ENV_PREFIX}/bin/accelerate"
DIFIX_OUTPUT_DIR = "/content/outputs/difix_mvsgaussian_finetune"
DIFIX_DATA_JSON = "/content/Difix3D/data/mvsgaussian_data.json"

cmd = [
    ACCELERATE_BIN, "launch",
    "--mixed_precision=fp16",
    "src/train_difix.py",
    f"--output_dir={DIFIX_OUTPUT_DIR}",
    f"--dataset_path={DIFIX_DATA_JSON}",
    "--pretrained_model_name_or_path=stabilityai/stable-diffusion-2-1-base",
    "--gradient_checkpointing",
    "--set_grads_to_none",
    "--max_train_steps=1000",
    "--train_batch_size=1",
    "--dataloader_num_workers=2",
    "--checkpointing_steps=500",
    "--eval_freq=250",
    "--viz_freq=100",
    "--tracker_run_name=run_alpha",
    "--timestep=199",
]

env = os.environ.copy()
env["PATH"] = f"/usr/local/cuda-11.8/bin:{DIFIX_ENV_PREFIX}/bin:" + env.get("PATH", "")
env["PYTHONPATH"] = f"/content/Difix3D:{DIFIX_ENV_PREFIX}/lib/python3.10/site-packages"

env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print(f"Đang Fine-tune mô hình (1000 steps, size 384)... Hãy xem nó vượt qua mốc Step 2 mượt mà!")
process = subprocess.Popen(
    cmd, cwd="/content/Difix3D",
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=env
)

for line in process.stdout:
    print(line, end="")

if process.wait() != 0:
    raise RuntimeError("Fine-tune thất bại.")

print(f"\nXONG BƯỚC 8-difix(e)! Mô hình của bạn đã được Fine-tune thành công!")

=== BƯỚС 8-difix(e): HẠ RESOLUTION XUỐNG 384 & CHỐNG PHÂN MẢNH RAM ===
✅ Đã giảm độ phân giải xuống 384x384 để triệt tiêu hoàn toàn nguy cơ OOM!
🚀 Đang Fine-tune mô hình (1000 steps, size 384)... Hãy xem nó vượt qua mốc Step 2 mượt mà!
The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_processes` was set to a value of `1`
	`--num_machines` was set to a value of `1`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
/usr/local/miniconda/envs/difix/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /usr/local/miniconda/envs/difix/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `

## Step 5 - Inference and Zero shot pretrained baseline comparison

In [ ]:
import subprocess, glob, os

print("=== BƯỚС 8-difix(f) [SỬA v2]: Inference, base nvidia/difix + checkpoint fine-tune thật ===")

DIFIX_PYTHON_BIN = "/usr/local/miniconda/envs/difix/bin/python"
DIFIX_OUTPUT_DIR = "/content/outputs/difix_mvsgaussian_finetune"
INPUT_DIR = "/content/SplatFormer/outputs/ood_eval_mvsinit_alpha/test/default"
OUT_DIR = "/content/difix_final_results_v2"

ckpts = sorted(glob.glob(os.path.join(DIFIX_OUTPUT_DIR, "checkpoints", "*.pkl")), key=os.path.getmtime)
if not ckpts:
    raise RuntimeError(f" Không tìm thấy checkpoint .pkl nào trong {DIFIX_OUTPUT_DIR}/checkpoints -- "
                        f"BƯỚС8-difix(e) [SỬA] có thực sự chạy xong không?")
latest_ckpt = ckpts[-1]
print(f"-> Dùng checkpoint fine-tune: {latest_ckpt}")

inference_script = f'''
import os, glob, sys, torch
from PIL import Image

sys.path.insert(0, "/content/Difix3D/src")
from model import Difix
from pipeline_difix import DifixPipeline

CKPT_PATH = "{latest_ckpt}"
INPUT_DIR = "{INPUT_DIR}"
OUT_DIR = "{OUT_DIR}"
os.makedirs(OUT_DIR, exist_ok=True)

print("-> Khởi tạo Difix() rồi nạp base nvidia/difix thật (giống hệt lúc train)...")
model = Difix(timestep=199)   # construct mặc định (sd-turbo + LoRA rỗng) -- sẽ bị GHI ĐÈ ngay dưới
pipe = DifixPipeline.from_pretrained("nvidia/difix", trust_remote_code=True)
model.unet = pipe.unet.to("cuda")
model.vae = pipe.vae.to("cuda")
model.vae.decoder.gamma = 1
del pipe
torch.cuda.empty_cache()

print("-> Overlay checkpoint fine-tune lên trên base vừa nạp...")
sd = torch.load(CKPT_PATH, map_location="cpu")
_sd_unet = model.unet.state_dict()
for k in sd["state_dict_unet"]:
    _sd_unet[k] = sd["state_dict_unet"][k]
model.unet.load_state_dict(_sd_unet)
_sd_vae = model.vae.state_dict()
for k in sd["state_dict_vae"]:
    _sd_vae[k] = sd["state_dict_vae"][k]
model.vae.load_state_dict(_sd_vae)
model.set_eval()
n_unet_keys = len(sd["state_dict_unet"])
n_vae_keys = len(sd["state_dict_vae"])
print(f"   Đã overlay {{n_unet_keys}} key UNet, {{n_vae_keys}} key VAE từ checkpoint.")

pred_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*_pred.png")))
if not pred_files:
    print(f"CẢNH BÁO: không tìm thấy ảnh *_pred.png nào trong {{INPUT_DIR}}")
else:
    print(f"-> Sửa {{len(pred_files)}} ảnh...")
    for p in pred_files:
        name = os.path.basename(p)
        img = Image.open(p).convert("RGB")
        w = img.width - img.width % 8
        h = img.height - img.height % 8
        out = model.sample(img, width=w, height=h, prompt="remove degradation")
        save_path = os.path.join(OUT_DIR, name.replace("_pred.png", "_difix_v2.png"))
        out.save(save_path)
        print(f"  Đã lưu: {{save_path}}")

print("\\nXONG. Ảnh đã lưu ở:", OUT_DIR)
'''

script_path = "/content/run_difix_inference_v2.py"
with open(script_path, "w") as f:
    f.write(inference_script)

print("Đang chạy inference...")
res = subprocess.run([DIFIX_PYTHON_BIN, script_path], text=True, capture_output=True)
print(res.stdout)
if res.returncode != 0:
    raise RuntimeError(f" Inference thất bại:\n{res.stderr[-3000:]}")

print(f"\nXong. So ảnh ở {OUT_DIR} với 2 lần trước -- lần này phải còn giữ được dáng vật thể rõ ràng.")

=== BƯỚС 8-difix(f) [SỬA v2]: Inference, base nvidia/difix + checkpoint fine-tune thật ===
-> Dùng checkpoint fine-tune: /content/outputs/difix_mvsgaussian_finetune/checkpoints/model_1.pkl
🚀 Đang chạy inference...
-> Khởi tạo Difix() rồi nạp base nvidia/difix thật (giống hệt lúc train)...
Initializing model with random weights
Number of trainable parameters in UNet: 865.91M
Number of trainable parameters in VAE: 0.52M
-> Overlay checkpoint fine-tune lên trên base vừa nạp...
   Đã overlay 686 key UNet, 90 key VAE từ checkpoint.
-> Sửa 10 ảnh...
  Đã lưu: /content/difix_final_results_v2/scene0_difix_v2.png
  Đã lưu: /content/difix_final_results_v2/scene1_difix_v2.png
  Đã lưu: /content/difix_final_results_v2/scene2_difix_v2.png
  Đã lưu: /content/difix_final_results_v2/scene3_difix_v2.png
  Đã lưu: /content/difix_final_results_v2/scene4_difix_v2.png
  Đã lưu: /content/difix_final_results_v2/scene5_difix_v2.png
  Đã lưu: /content/difix_final_results_v2/scene6_difix_v2.png
  Đã lưu: /conte

### Ver 2: Inference by nvidia/difix (no fine-tuning)

In [ ]:
import subprocess, os

print("=== INFERENCE nvidia/difix CHUẨN XÁC VỚI PIPELINE + TRUST_REMOTE_CODE ===")

DIFIX_PYTHON_BIN = "/usr/local/miniconda/envs/difix/bin/python"
INPUT_DIR = "/content/SplatFormer/outputs/ood_eval_mvsinit_alpha/test/default"
OUT_DIR = "/content/difix_pretrained_only_results"

inference_script = f'''
import os, glob, sys, torch
from PIL import Image

sys.path.insert(0, "/content/Difix3D/src")
from pipeline_difix import DifixPipeline

INPUT_DIR = "{INPUT_DIR}"
OUT_DIR = "{OUT_DIR}"
os.makedirs(OUT_DIR, exist_ok=True)

print("-> Nạp nvidia/difix với trust_remote_code=True...")
pipe = DifixPipeline.from_pretrained("nvidia/difix", trust_remote_code=True).to("cuda")

pred_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*_pred.png")))
if not pred_files:
    print(f"CẢNH BÁO: không tìm thấy ảnh *_pred.png nào trong {{INPUT_DIR}}")
else:
    print(f"-> Đang xử lý {{len(pred_files)}} ảnh...")
    for p in pred_files:
        name = os.path.basename(p)
        img = Image.open(p).convert("RGB")
        out = pipe(
            "remove degradation", image=img,
            num_inference_steps=1, timesteps=[199], guidance_scale=0.0,
        ).images[0]
        save_path = os.path.join(OUT_DIR, name.replace("_pred.png", "_difix_pretrained.png"))
        out.save(save_path)
        print(f"  Đã lưu: {{save_path}}")

print("\\nXONG. Ảnh đã lưu ở:", OUT_DIR)
'''

script_path = "/content/run_difix_pretrained_only.py"
with open(script_path, "w") as f:
    f.write(inference_script)

print("🚀 Đang chạy inference (nvidia/difix gốc)...")
res = subprocess.run([DIFIX_PYTHON_BIN, script_path], text=True, capture_output=True)
print(res.stdout)
if res.returncode != 0:
    raise RuntimeError(f"❌ Inference thất bại:\n{res.stderr[-3000:]}")

print(f"\n✅ Hoàn tất! Ảnh kết quả đã được lưu tại: {OUT_DIR}")

=== INFERENCE nvidia/difix CHUẨN XÁC VỚI PIPELINE + TRUST_REMOTE_CODE ===
🚀 Đang chạy inference (nvidia/difix gốc)...
-> Nạp nvidia/difix với trust_remote_code=True...
-> Đang xử lý 10 ảnh...
  Đã lưu: /content/difix_pretrained_only_results/scene0_difix_pretrained.png
  Đã lưu: /content/difix_pretrained_only_results/scene1_difix_pretrained.png
  Đã lưu: /content/difix_pretrained_only_results/scene2_difix_pretrained.png
  Đã lưu: /content/difix_pretrained_only_results/scene3_difix_pretrained.png
  Đã lưu: /content/difix_pretrained_only_results/scene4_difix_pretrained.png
  Đã lưu: /content/difix_pretrained_only_results/scene5_difix_pretrained.png
  Đã lưu: /content/difix_pretrained_only_results/scene6_difix_pretrained.png
  Đã lưu: /content/difix_pretrained_only_results/scene7_difix_pretrained.png
  Đã lưu: /content/difix_pretrained_only_results/scene8_difix_pretrained.png
  Đã lưu: /content/difix_pretrained_only_results/scene9_difix_pretrained.png

XONG. Ảnh đã lưu ở: /content/difix_pr

## Step 6 - Quantitative evaluation (PSNR, SSIM, LPIPS)

In [ ]:
import os, subprocess

PRED_DIR  = "/content/SplatFormer/outputs/ood_eval_mvsinit_alpha/test/default"  # *_pred.png (input) + *_gt.png
DIFIX_DIR = "/content/difix_pretrained_only_results"                            # Thư mục chứa *_difix_pretrained.png
CSV_OUT   = "/content/metrics_difix_pretrained_vs_splatformer.csv"
CONDA_PYTHON = "/usr/local/miniconda/envs/splatformer/bin/python"

eval_script = r'''
import os, sys, glob, csv
import numpy as np
from PIL import Image
import torch
import lpips
from skimage.metrics import structural_similarity as ssim_fn
from skimage.metrics import peak_signal_noise_ratio as psnr_fn

pred_dir, difix_dir, csv_out = sys.argv[1], sys.argv[2], sys.argv[3]
device = "cuda" if torch.cuda.is_available() else "cpu"
lpips_fn = lpips.LPIPS(net="vgg").to(device).eval()

def load(path, size=None):
    img = Image.open(path)
    if img.mode == "RGBA":
        bg = Image.new("RGB", img.size, (255, 255, 255))
        bg.paste(img, mask=img.split()[3])
        img = bg
    else:
        img = img.convert("RGB")
    if size is not None and img.size != size:
        img = img.resize(size, Image.LANCZOS)
    return np.asarray(img).astype(np.float32) / 255.0

def score(gt, pr):
    p = psnr_fn(gt, pr, data_range=1.0)
    s = ssim_fn(gt, pr, channel_axis=-1, data_range=1.0)
    t = lambda a: torch.from_numpy(a).permute(2, 0, 1).unsqueeze(0).to(device) * 2.0 - 1.0
    with torch.no_grad():
        l = lpips_fn(t(gt), t(pr)).item()
    return float(p), float(s), float(l)

# Tìm file kết quả của bản Pretrained
difix_files = sorted(glob.glob(os.path.join(difix_dir, "*_difix_pretrained.png")))
if not difix_files:
    raise SystemExit(f"Không thấy *_difix_pretrained.png nào trong {difix_dir} -- kiểm tra lại bước inference trước.")

rows = []
for f in difix_files:
    name = os.path.basename(f).replace("_difix_pretrained.png", "")
    gt_p, in_p = os.path.join(pred_dir, f"{name}_gt.png"), os.path.join(pred_dir, f"{name}_pred.png")
    if not (os.path.exists(gt_p) and os.path.exists(in_p)):
        print(f"[bỏ qua] {name}: thiếu {name}_gt.png hoặc {name}_pred.png trong {pred_dir}")
        continue
    gt_img = Image.open(gt_p)
    size = gt_img.size
    gt = load(gt_p)
    base = score(gt, load(in_p, size))
    dfx  = score(gt, load(f, size))
    rows.append((name, base, dfx))

if not rows:
    raise SystemExit("Không ghép được cặp nào (Difix output / GT / input).")

def mean(i, k):
    return float(np.mean([r[i][k] for r in rows]))

print("=" * 86)
print(f"{'Scene':<14}{'':<4}{'PSNR(dB)':>10}{'SSIM':>9}{'LPIPS':>9}   |  {'dPSNR':>7}{'dSSIM':>8}{'dLPIPS':>8}")
print("-" * 86)
for name, b, d in rows:
    print(f"{name[:14]:<14}{'in':<4}{b[0]:>10.2f}{b[1]:>9.4f}{b[2]:>9.4f}   |")
    print(f"{'':<14}{'out':<4}{d[0]:>10.2f}{d[1]:>9.4f}{d[2]:>9.4f}   |  {d[0]-b[0]:>+7.2f}{d[1]-b[1]:>+8.4f}{d[2]-b[2]:>+8.4f}")
print("-" * 86)
mb = [mean(1, k) for k in range(3)]
md = [mean(2, k) for k in range(3)]
print(f"TRUNG BÌNH ({len(rows)} ảnh) - DIFIX PRETRAINED (GỐC)")
print(f"  SplatFormer (input) : PSNR {mb[0]:.2f} | SSIM {mb[1]:.4f} | LPIPS {mb[2]:.4f}")
print(f"  Difix Pretrained    : PSNR {md[0]:.2f} | SSIM {md[1]:.4f} | LPIPS {md[2]:.4f}")
print(f"  Chênh lệch          : PSNR {md[0]-mb[0]:+.2f} | SSIM {md[1]-mb[1]:+.4f} | LPIPS {md[2]-mb[2]:+.4f}")
print("=" * 86)

with open(csv_out, "w", newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["scene", "stage", "PSNR", "SSIM", "LPIPS"])
    for name, b, d in rows:
        w.writerow([name, "splatformer_input", round(b[0], 4), round(b[1], 4), round(b[2], 4)])
        w.writerow([name, "difix_pretrained_output", round(d[0], 4), round(d[1], 4), round(d[2], 4)])
print("Đã lưu bảng điểm tại:", csv_out)
'''

script_path = "/content/run_difix_pretrained_metrics.py"
with open(script_path, "w") as f:
    f.write(eval_script)

res = subprocess.run([CONDA_PYTHON, script_path, PRED_DIR, DIFIX_DIR, CSV_OUT], text=True, capture_output=True)
print(res.stdout)
if res.returncode != 0:
    print("Lỗi:\n", res.stderr[-3000:])

Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
Loading model from: /usr/local/miniconda/envs/splatformer/lib/python3.10/site-packages/lpips/weights/v0.1/vgg.pth
Scene               PSNR(dB)     SSIM    LPIPS   |    dPSNR   dSSIM  dLPIPS
--------------------------------------------------------------------------------------
scene0        in       13.86   0.7693   0.2574   |
              out      13.64   0.6233   0.2716   |    -0.22 -0.1460 +0.0142
scene1        in       14.04   0.5664   0.3752   |
              out      14.08   0.5348   0.3706   |    +0.04 -0.0316 -0.0046
scene2        in       12.22   0.6649   0.2975   |
              out      12.19   0.5583   0.3008   |    -0.03 -0.1066 +0.0033
scene3        in       13.93   0.7284   0.2762   |
              out      13.66   0.5691   0.2837   |    -0.27 -0.1593 +0.0075
scene4        in       11.45   0.6615   0.3199   |
              out      11.52   0.5652   0.3122   |    +0.07 -0.0963 -0.0077
scene5        in 

## Step 7 - Interactive 3D Gaussian point cloud viewer

In [ ]:
import os, glob, torch
import numpy as np
from IPython.display import HTML, display

print("=== TRÍCH XUẤT VÀ VẼ SƠ ĐỒ 3D TƯƠNG TÁC TỪ DỮ LIỆU SPLATFORMER ===")


ckpt_candidates = sorted(glob.glob("/content/mvsinit_alpha_nerfstudio_subset/**/splatfacto/nerfstudio_models/*.ckpt", recursive=True))

if not ckpt_candidates:
    ckpt_candidates = sorted(glob.glob("/content/nerfstudio_out_mvsinit_alpha/**/*.ckpt", recursive=True))

if not ckpt_candidates:
    raise RuntimeError(" Không tìm thấy file checkpoint .ckpt nào trong thư mục nerfstudio.")

target_ckpt = ckpt_candidates[0]
print(f"📦 Đang trích xuất dữ liệu 3D từ: {target_ckpt}")

data = torch.load(target_ckpt, map_location="cpu", weights_only=False)
state = data.get("pipeline", data)

means, features_dc, opacities = None, None, None

for k, v in state.items():
    if "means" in k:
        means = v.detach().float().numpy()
    elif "features_dc" in k:
        features_dc = v.detach().float().numpy()
    elif "opacities" in k:
        opacities = torch.sigmoid(v).detach().float().numpy()

if means is None or features_dc is None:
    raise RuntimeError(" Không tìm thấy trường 'means' hoặc 'features_dc' trong checkpoint.")

if opacities is not None:
    mask = (opacities.squeeze() > 0.05)
    means = means[mask]
    features_dc = features_dc[mask]

SH_C0 = 0.28209479177387814
rgb = np.clip(features_dc.squeeze() * SH_C0 + 0.5, 0.0, 1.0)


if len(means) > 30000:
    indices = np.random.choice(len(means), 30000, replace=False)
    means = means[indices]
    rgb = rgb[indices]

center = np.mean(means, axis=0)
means = means - center
scale_factor = np.max(np.abs(means))
means = means / (scale_factor if scale_factor > 0 else 1.0)

positions_js = means.flatten().tolist()
colors_js = rgb.flatten().tolist()

print(f"Đã trích xuất thành công {len(means)} điểm 3D Gaussian!")

viewer_html = f"""
<div id="container-3d" style="width: 100%; height: 550px; background-color: #121212; border-radius: 8px; position: relative;">
    <div style="position: absolute; top: 12px; left: 16px; color: #00ffcc; font-family: monospace; font-size: 13px; z-index: 10; pointer-events: none; background: rgba(0,0,0,0.6); padding: 6px 12px; border-radius: 4px;">
        🕹️ <b>Sơ đồ 3D SplatFormer:</b> Chuột trái: Xoay 360° | Chuột phải: Di chuyển | Cuộn chuột: Zoom
    </div>
</div>

<script src="https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js"></script>
<script src="https://cdn.jsdelivr.net/npm/three@0.128.0/examples/js/controls/OrbitControls.js"></script>

<script>
(function() {{
    const container = document.getElementById('container-3d');
    const scene = new THREE.Scene();
    scene.background = new THREE.Color(0x121212);

    const camera = new THREE.PerspectiveCamera(50, container.clientWidth / container.clientHeight, 0.01, 100);
    camera.position.set(1.5, 1.2, 2.0);

    const renderer = new THREE.WebGLRenderer({{ antialias: true }});
    renderer.setSize(container.clientWidth, container.clientHeight);
    renderer.setPixelRatio(window.devicePixelRatio);
    container.appendChild(renderer.domElement);

    const controls = new THREE.OrbitControls(camera, renderer.domElement);
    controls.enableDamping = true;
    controls.dampingFactor = 0.05;

    // Lưới mặt sàn tham chiếu không gian
    const grid = new THREE.GridHelper(3, 20, 0x444444, 0x222222);
    grid.position.y = -0.8;
    scene.add(grid);

    // Dựng Geometry từ các điểm XYZ và RGB của SplatFormer
    const geometry = new THREE.BufferGeometry();
    const positions = new Float32Array({positions_js});
    const colors = new Float32Array({colors_js});

    geometry.setAttribute('position', new THREE.BufferAttribute(positions, 3));
    geometry.setAttribute('color', new THREE.BufferAttribute(colors, 3));

    const material = new THREE.PointsMaterial({{
        size: 0.02,
        vertexColors: true,
        transparent: true,
        opacity: 0.85
    }});

    const pointCloud = new THREE.Points(geometry, material);
    scene.add(pointCloud);

    function animate() {{
        requestAnimationFrame(animate);
        controls.update();
        renderer.render(scene, camera);
    }}
    animate();

    window.addEventListener('resize', () => {{
        camera.aspect = container.clientWidth / container.clientHeight;
        camera.updateProjectionMatrix();
        renderer.setSize(container.clientWidth, container.clientHeight);
    }});
}})();
</script>
"""

display(HTML(viewer_html))

=== TRÍCH XUẤT VÀ VẼ SƠ ĐỒ 3D TƯƠNG TÁC TỪ DỮ LIỆU SPLATFORMER ===
📦 Đang trích xuất dữ liệu 3D từ: /content/mvsinit_alpha_nerfstudio_subset/02691156-2628b6cfcf1a53465569af4484881d20-10_mvsinit_alpha/splatfacto/nerfstudio_models/step-000001999.ckpt
✅ Đã trích xuất thành công 5801 điểm 3D Gaussian!
